# ParaLoRA embedding extraction

Extracts per-residue ProtT5 embeddings (after LoRA fine-tuning) and stores
them as one `.npz` per split. The embeddings are consumed by the structure
branch as node features and by external classifiers.


In [ ]:
import os, json, numpy as np
from paralora.evaluate import extract_embeddings


In [ ]:
with open('configs/paralora.json') as f:
    cfg = json.load(f)
cfg['model_name_or_path'] = os.environ.get('PROTT5_PATH', cfg['model_name_or_path'])


In [ ]:
embeddings = extract_embeddings(
    config=cfg,
    checkpoint_path='runs/paralora/trainable_params.pt',
    split_path='data/paralora/test.csv',
    batch_size=8,
)
print('Extracted embeddings for', len(embeddings), 'sequences; example shape:', embeddings[0].shape)


In [ ]:
os.makedirs('runs/paralora/embeddings', exist_ok=True)
flat = np.concatenate(embeddings, axis=0)
lengths = np.array([e.shape[0] for e in embeddings])
names = np.array(list(range(len(embeddings))))
np.savez('runs/paralora/embeddings/test.npz', names=names, lengths=lengths, embeddings=flat)
print('Saved runs/paralora/embeddings/test.npz')
